# DMOSpeech 2: Preliminary Test of Noise-Aware Guidance at the Handover
## Prompt-Informed Environment Cancellation (PIEC), SNR-routed prompt enhancement, SWC, SCG, SNNG and an antithetic student pair

**Why.** EXP-013/014 showed that most noisy-prompt failures at an early handover are **speaker-similarity** failures: the prompt's noise leaks into the generated voice. No zero-cost change at the handover fixed that. These methods act on the cause.

| Method | Formula | Cost at Step 4 |
|---|---|---|
| **SNNG** | Build a noise-only prompt from the prompt's own **quietest 20% of frames**. Each teacher step adds one pass with that prompt and steers away from it: **v = v_c + γ(v_c − v_u) + η(v_c − v_noise)**, γ = 2 (usual CFG), η = 0.5 and 1.0 | 3 passes × 5 steps + 3 = **18 NFE** |
| **SNR-Compensated Guidance (SCG)** | Prompt noise shrinks the guidance gap g = v_c − v_u (median 0.236 clean, 0.116 at 15 dB). The guidance scale is raised by the factor the prompt SNR predicts: **γ(Ŝ) = 2 · clip(0.236 / (−0.0263 + 0.0078·Ŝ), 1, r_max)**, with r_max = 2.5 (full) and 1.5 (capped, γ ≤ 3) | **13 NFE** (no extra pass) |
| **Prompt-Informed Environment Cancellation (PIEC)** ⭐ new | Zero-shot TTS copies the prompt's acoustic environment into the output. PIEC estimates that environment from the prompt itself (per mel band, the mean of the prompt's quietest 20% of frames) and cancels it from the **generated** frames before the vocoder: **G = clip(1 − β(Ŝ)·N_b / M, 0.1, 1)**, **β(Ŝ) = clip((30 − Ŝ)/10, 0, 1)**. `piec4`: on the plain Step-4 output; `swcpiec4`: SWC conditioning, then PIEC on its output | **13 NFE** (no network cost) |
| **Controls** (`enh8`, `piec8`) | The same cleaning on the full Step-8 output. The report judges `enh4` against `enh8` and `piec4` against `piec8`, to show whether the early handover is safe *within* the cleaned system, not only better than the uncleaned reference | 21 NFE |
| **SNR-routed prompt enhancement** ⭐ | The voice sample is cleaned by a small pretrained speech enhancer (DNS64 Demucs) before the teacher and student see it. `enh4`: full cleaning, handover at Step 4; `enh6`: full cleaning, Step 6; `enhs4`: cleaning strength set by SNR (none at 30 dB or above, full at 20 dB or below), Step 4. The router policy `*_heavy` sends clean prompts to plain Step 4, heavy noise to the enhanced path and mild noise to Step 8, with both SNR cut-offs calibrated by Learn-Then-Test | **13 / 17 NFE** (+ about 0.2 NFE-equivalent for the enhancer, timed separately) |
| **SNR-Wiener Conditioning (SWC)** | Removes the prompt's noise from the conditioning before the teacher and student see it. Per mel band, the noise floor N_b is the mean of the prompt's quietest 20% of frames; each prompt frame gets the gain **G = clip(1 − β(Ŝ)·N_b / M, 0.1, 1)**, with **β(Ŝ) = β_max · clip((30 − Ŝ)/10, 0, 1)**: no cleaning at 30 dB or above, full strength at 20 dB or below. β_max = 1.0 and 0.5 | **13 NFE** (no extra pass) |
| **Antithetic pair** | From the plain Step-4 endpoint, run the student with its noise ε and with −ε; keep the output closer to the teacher's endpoint | 13 + 3 = **16 NFE** |

The reference (Step 8) costs 21 NFE, so each method saves compute **if** it makes noisy prompts safe at Step 4. In the route policy, **the prompt SNR decides** which inputs use plain Step 4, which use the method, and which go to Step 8.

**What it runs:** the EXP-012 Quick set (60 development speakers × clean, noisy15, Gaussian 18/20/22/25 dB = 360 cases). Stored plain Step-4 and Step-8 results are reused; only the new outputs are synthesized and scored. About **2.5–3 hours on an A100** or about **6.5–7 hours on a T4** (estimates). Development speakers only; settings, pass/fail criteria and the rule for picking the best method are written to `preregistration.json` before any synthesis. Results are flushed to Google Drive after every case, so the run resumes after a disconnect.

---
### Instructions
1. Set the runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`).
2. **Runtime > Run all.** Step 6 runs in the background; if the browser disconnects, reconnect and run **Step 6b**.
3. The last cell downloads `DMOSpeech2_Noise_Guidance_Results.zip`.

### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi

### Step 2: Clone Repositories & Install Dependencies
`Shonget-DMOSpeech2` is checked out at `8a7868f`, the commit whose `core/` sampler produced the T4 grid (it also contains the `656d86f` fix that makes SIM honour the ground-truth sample rate). The thesis repository (branch `rhenelAudit`) supplies the runner, the split manifests and the T4 grid; if it is private, add a Colab secret named `GITHUB_TOKEN`.

If Colab shows **"Restart session"** after the installs, click it, then continue from **Step 3** (no need to re-run Step 2). Every later step re-enters the repository folder and re-reads its settings, so a restart is safe.

In [ ]:
import os, subprocess
DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}
!git log --oneline -1

token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
THESIS_BRANCH = 'rhenelAudit'   # the branch that holds the follow-up runner
if not os.path.exists('/content/Final-Thesis-Setup'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git",
                    '/content/Final-Thesis-Setup'], check=True)
!git -C /content/Final-Thesis-Setup fetch -q origin {THESIS_BRANCH} && git -C /content/Final-Thesis-Setup checkout -q {THESIS_BRANCH} && git -C /content/Final-Thesis-Setup pull -q origin {THESIS_BRANCH}
!git -C /content/Final-Thesis-Setup log --oneline -1

# Same pinned dependencies as the 1,000-utterance GPU retest notebook
!pip install -q "numpy==2.2.3" "pandas==2.2.3"
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

import numpy as np, pandas as pd, numba
print(f"Verified: NumPy {np.__version__} | Pandas {pd.__version__} | Numba {numba.__version__} [OK]")

runner_path = '/content/Final-Thesis-Setup/code/colab/run_followup_tests.py'
if not os.path.exists(runner_path):
    raise FileNotFoundError(f"{runner_path} is not on branch {THESIS_BRANCH}. Push it to GitHub before continuing.")

### Step 3: Download DMOSpeech 2 Checkpoints
`model_85000.pt` (4.0 GB, teacher + student) and `model_1500.pt` (270 MB, duration predictor) into `ckpts/`, checked against the SHA-256 hashes recorded for the earlier runs.

In [ ]:
%cd /content/Shonget-DMOSpeech2
!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

import hashlib
for name, digest in {'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
                     'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'}.items():
    h = hashlib.sha256()
    with open(f'ckpts/{name}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    assert h.hexdigest() == digest, f"{name}: SHA-256 mismatch (delete it and re-run this step)"
    print(f"{name}: SHA-256 verified")

### Step 4: Download the Seed-TTS test-en Dataset
Prompt and ground-truth audio for all 666 speakers: the 500 in the splits and the 166 outside them.

In [ ]:
import os
os.chdir('/content/Shonget-DMOSpeech2')

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'

# Check if audio is missing or only a Git LFS pointer
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    print("Resolving Git LFS binaries for the Seed-TTS evaluation dataset...")
    if os.path.exists('data/seed-tts-eval/.git'):
        !cd data/seed-tts-eval && git lfs install && git lfs pull
    if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
        print("Cleaning leftover pointer files and downloading via huggingface_hub...")
        !rm -rf data/seed-tts-eval
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir="data/seed-tts-eval")
    print("Dataset setup complete!")
else:
    print("Seed-TTS audio binaries verified!")

!ls -lh data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav
!wc -l data/seed-tts-eval/en/meta.lst

### Step 5: Smoke Test + Reproduction Check (2 development speakers, a few minutes)
Runs every method (PIEC, prompt enhancement, SWC, SCG, SNNG, antithetic pair) and the controls for 2 non-pilot development speakers (clean and Gaussian 20 dB). Checks that the guided teacher loop with γ = 2 and η = 0 gives exactly the plain teacher endpoint, and re-scores the plain Step-4 and Step-8 outputs to check that they match the stored scores (the stored labels are reused in the analysis). Writes to a **separate** `smoke/` folder.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'noise_guidance_prelim')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_noise_guidance_prelim.py"'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

!pip install -q --no-deps denoiser==0.1.5
!python {RUNNER} smoke --out "{OUT}" --dmo_root {DMO}

import pandas as pd
ex = pd.read_csv(f'{OUT}/smoke/snng.csv')
print(f"Smoke test OK: {len(ex)} cases")
display(ex[['utt_id', 'condition', 'prompt_snr', 'd_4', 'd4_stored', 'noise_frames', 'anti_pick', 'beta_piec', 'snr_after_enh', 'enh_ms', 'is_failure_piec4', 'is_failure_swcpiec4', 'is_failure_enh4', 'is_failure_enhs4', 'is_failure_swc', 'is_failure_snng10']])

### Step 6: Full Run + Analysis
Writes `preregistration.json` (once), runs every method for all 360 cases, then runs the analysis.

The job runs **in the background** and writes its full output to `MyDrive/dmospeech2_results/noise_guidance_log.txt`; this cell prints one progress line per minute. If only the browser disconnects, the job keeps running; reconnect and run **Step 6b**.

**If the Colab machine itself is reset**, the results so far are safe in Google Drive (`MyDrive/dmospeech2_results/noise_guidance_prelim/`). Re-run Steps 1–4, skip Step 5, then run this cell again: it continues from the last saved case.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'noise_guidance_prelim')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_noise_guidance_prelim.py"'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

!pip install -q --no-deps denoiser==0.1.5
import subprocess, time, re
log_path = os.path.join(persist, 'noise_guidance_log.txt')
cmd = f'python {RUNNER} run --out "{OUT}" --dmo_root {DMO}'
running = subprocess.run(['pgrep', '-f', 'run_noise_guidance_prelim.py'], capture_output=True, text=True).stdout.split()
if running:
    pid = int(running[0])
    alive = lambda: os.path.exists(f'/proc/{pid}')
    print(f"A job is already running (pid {pid}); monitoring it instead of starting a new one.")
else:
    proc = subprocess.Popen(f'nohup {cmd} >> "{log_path}" 2>&1', shell=True)
    alive = lambda: proc.poll() is None
    print(f"Started (pid {proc.pid}). Full log: {log_path}")

last = ''
while alive():
    time.sleep(60)
    try:
        lines = open(log_path, encoding='utf-8', errors='ignore').read().splitlines()
        prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
        if prog and prog[-1] != last:
            last = prog[-1]
            print(time.strftime('%H:%M'), last)
    except FileNotFoundError:
        pass
print(f"Finished{'' if running else f' with exit code {proc.returncode}'}. Last lines of the log:")
for line in open(log_path, encoding='utf-8', errors='ignore').read().splitlines()[-15:]:
    print(line)

### Step 6b: Check Progress After a Reconnect
Shows whether the background job is still running and how many cases are saved. Does **not** start a second job.

In [ ]:
import os, re, subprocess
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'noise_guidance_prelim')
log_path = os.path.join(persist, 'noise_guidance_log.txt')
running = subprocess.run(['pgrep', '-f', 'run_noise_guidance_prelim.py'], capture_output=True, text=True).stdout.split()
print("Job running:", "yes (pid " + ", ".join(running) + ")" if running else "no")
if os.path.exists(log_path):
    lines = open(log_path, encoding='utf-8', errors='ignore').read().splitlines()
    prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
    print("Latest progress:", prog[-1] if prog else "(none yet)")
    for line in lines[-5:]:
        print(line)
path = os.path.join(OUT, 'snng.csv')
n = sum(1 for _ in open(path, encoding='utf-8')) - 1 if os.path.exists(path) else 0
print(f"snng.csv: {n} of 360 cases saved")

### Step 7: View the Report
If the run stopped early, add `--allow_partial` for an interim look (the report is then marked PARTIAL).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'noise_guidance_prelim')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_noise_guidance_prelim.py"'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

from IPython.display import Markdown, display
rep = os.path.join(OUT, 'NOISE_GUIDANCE_REPORT.md')
if not os.path.exists(rep) and os.path.exists(os.path.join(OUT, 'snng.csv')):
    !python {RUNNER} analyze --out "{OUT}"
if os.path.exists(rep):
    display(Markdown(open(rep, encoding='utf-8').read()))
else:
    print(f"No report yet at {rep}. Run Step 6 first.")

### Step 8: Zip All Outputs and Download
Packs the CSV, preregistration, results JSON, report and the runners into one ZIP, also saved to Drive.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'noise_guidance_prelim')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_noise_guidance_prelim.py"'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

import zipfile, subprocess, datetime
zip_name = 'DMOSpeech2_Noise_Guidance_Results.zip'
with open(os.path.join(OUT, 'RUN_INFO.txt'), 'w') as f:
    f.write(f"dmospeech2_commit: {subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}\n")
    f.write(f"thesis_commit: {subprocess.run(['git', '-C', THESIS, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.utcnow().isoformat()}Z\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_ in os.walk(OUT):
        dirs[:] = [d for d in dirs if d != 'listening']
        for fname in files_:
            if fname.startswith('tmp_') or fname.endswith('.wav'):
                continue
            full = os.path.join(root, fname)
            zf.write(full, os.path.relpath(full, OUT))
    for script in ('run_noise_guidance_prelim.py', 'run_handover_operators_prelim.py', 'run_endpoint_extrapolation_prelim.py', 'run_consistency_gate_prelim.py', 'run_followup_tests.py'):
        zf.write(f'{THESIS}/code/colab/{script}', f'code/{script}')
print(f"Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)")
!cp "{zip_name}" "{persist}/"
print(f"Also saved to Google Drive: {persist}/{zip_name}")
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f"Download manually from the Files panel: /content/Shonget-DMOSpeech2/{zip_name}")